<a href="https://colab.research.google.com/github/sayam-h069/Compiler_Design/blob/main/CD_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y flex

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison flex-doc
The following NEW packages will be installed:
  flex libfl-dev libfl2
0 upgraded, 3 newly installed, 0 to remove and 79 not upgraded.
Need to get 324 kB of archives.
After this operation, 1,148 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fetched 324 kB in 1s (311 kB/s)
Selecting previously unselected package flex.
(Reading dat

1(a)

In [2]:
%%writefile lab1a.l
%option noyywrap

%%
[A-Z][a-zA-Z]*              { printf("Capitalized word: %s\n", yytext); }
[a-z]+                      { printf("Lowercase word: %s\n", yytext); }
[0-9]+                      { printf("Integer: %s\n", yytext); }
[0-9]+\.[0-9]+              { printf("Decimal number: %s\n", yytext); }
[a-zA-Z_][a-zA-Z0-9_]*      { printf("Identifier: %s\n", yytext); }
[ \t\n]+                    ;
.                           { printf("Special character: %s\n", yytext); }
%%

int main(void)
{
    printf("Enter text:\n");
    yylex();
    return 0;
}

Writing lab1a.l


In [3]:
!flex lab1a.l
!gcc lex.yy.c -o lab1a
!echo "Apple 123 45.67 hello! _value" | ./lab1a

Enter text:
Capitalized word: Apple
Integer: 123
Decimal number: 45.67
Lowercase word: hello
Special character: !
Identifier: _value


1(b)

In [4]:
%%writefile lab1b.l
%option noyywrap

%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

void printNumberInWords(int n)
{
    char *ones[] = {
        "", "one", "two", "three", "four",
        "five", "six", "seven", "eight", "nine"
    };

    char *teens[] = {
        "ten", "eleven", "twelve", "thirteen", "fourteen",
        "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"
    };

    char *tens[] = {
        "", "", "twenty", "thirty", "forty",
        "fifty", "sixty", "seventy", "eighty", "ninety"
    };

    if (n >= 10 && n <= 19)
        printf("%d : %s\n", n, teens[n - 10]);
    else if (n >= 20 && n <= 99) {
        printf("%d : %s", n, tens[n / 10]);
        if (n % 10 != 0)
            printf(" %s", ones[n % 10]);
        printf("\n");
    }
}
%}

%%
[0-9]+  {
            if (strlen(yytext) == 2)
                printNumberInWords(atoi(yytext));
        }
.|\n    ;
%%

int main(void)
{
    printf("Enter numbers:\n");
    yylex();
    return 0;
}

Writing lab1b.l


In [5]:
!flex lab1b.l
!gcc lex.yy.c -o lab1b
!echo "10 25 40 99 7 123" | ./lab1b

Enter numbers:
10 : ten
25 : twenty five
40 : forty
99 : ninety nine


In [8]:
%%writefile lab1c.l
%option noyywrap

%{
#include <stdio.h>

int isLeapYear(int year)
{
    return (year % 400 == 0) || (year % 4 == 0 && year % 100 != 0);
}

int isValidDate(int day, int month, int year)
{
    int daysInMonth[] = {
        0, 31, 28, 31, 30, 31, 30,
        31, 31, 30, 31, 30, 31
    };

    if (year < 1 || month < 1 || month > 12 || day < 1)
        return 0;

    if (month == 2 && isLeapYear(year))
        return day <= 29;

    return day <= daysInMonth[month];
}
%}

%%
[0-9]{2}\/[0-9]{2}\/[0-9]{4} {
    int day, month, year;

    sscanf(yytext, "%d/%d/%d", &day, &month, &year);

    if (isValidDate(day, month, year))
        printf("%s : Valid date\n", yytext);
    else
        printf("%s : Invalid date\n", yytext);
}

[ \t\n]+  ;
.         ;
%%

int main(void)
{
    printf("Enter date(s) in DD/MM/YYYY format:\n");
    yylex();
    return 0;
}

Overwriting lab1c.l


In [9]:
!flex lab1c.l
!gcc lex.yy.c -o lab1c
!echo "29/02/2024 29/02/2023 31/04/2025 31/12/2025" | ./lab1c

Enter date(s) in DD/MM/YYYY format:
29/02/2024 : Valid date
29/02/2023 : Invalid date
31/04/2025 : Invalid date
31/12/2025 : Valid date
